In [1]:
from nova_py import MILAn, PArMesan
from transformers import PreTrainedTokenizerFast
import tensorflow as tf
import json
import numpy as np
import importlib

tokenizer = PreTrainedTokenizerFast.from_pretrained("bert-base-uncased")
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

2025-06-22 23:27:58.649260: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


0

In [2]:
llm = MILAn.Model(tokenizer)

In [3]:
tokens = llm.tokenizer(["hello there", "nice to meet you"], padding = True, return_tensors="tf")
print(tokens['input_ids'])

tf.Tensor(
[[ 101 7592 2045  102    0    0]
 [ 101 3835 2000 3113 2017  102]], shape=(2, 6), dtype=int32)


In [4]:
prompts = ["the world is yours", "snoop doggy dog"]
llm(prompts=prompts, token_limit=5)

/usr/local/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


['sriʂ limitation miles liberalism', '##kovich ᶜ oleg routed outdoor']

In [5]:
llm.PArM.warm

True

In [6]:
llm.PArM.summary()

Model: "PArM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ NERf (Model)                    │ ?                      │    67,155,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ PArMe (Layer)                   │ ?                      │       262,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ PArMf0 (Layer)                  │ ?                      │    50,352,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ PArMf1 (Layer)                  │ ?                      │    50,352,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_2 (Layer)                 │ ?                      │    62,539,578 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 230,661,050 (879.90 MB)

 Trainable params: 230,661,050 (879.90 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
def dummy_dataset(batch_size=2, seq_len=10, vocab_size=tokenizer.vocab_size):
    while True:
        x = np.random.randint(5, vocab_size, size=(batch_size, seq_len)).astype(np.int32)
        y = np.roll(x, shift=-1, axis=1)
        y[:, -1] = 102  # example stop token
        mask = (x != 0).astype(np.float32)
        yield (x, mask), y

train_ds = tf.data.Dataset.from_generator(
    dummy_dataset,
    output_signature=((tf.TensorSpec(shape=(None, None), dtype=tf.int32),
                       tf.TensorSpec(shape=(None, None), dtype=tf.float32)),
                      tf.TensorSpec(shape=(None, None), dtype=tf.int32))
)

In [ ]:
importlib.reload(MILAn)
importlib.reload(PArMesan)
# --- TRAINING LOOP ---
EPOCHS = 5
STEPS_PER_EPOCH = 100

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    for step, ((x_batch, mask_batch), y_batch) in enumerate(train_ds.take(STEPS_PER_EPOCH)):
        # print(x_batch)
        loss = llm.train(x_batch, mask_batch, y_batch)
        if step % 10 == 0:
            print(f"Step {step}: Loss = {loss.numpy():.4f}")

    # Eval preview
    prompts = ["the mitochondria is"]
    outputs = llm(prompts=prompts, token_limit=30)
    print(f"\nSample output: {outputs[0]}")


Epoch 1/5


In [ ]:
    # Eval preview
    prompts = ["the mitochondria is"]
    outputs = llm(prompts=prompts, token_limit=30)
    print(f"\nSample output: {outputs[0]}")